In [ ]:
# Montamos Google Drive: es donde estan tanto los modelos ya entrenados
# (modelos_exportados/) como el features.csv generado por el honeypot
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
# Nos situamos en la carpeta del proyecto y listamos su contenido para
# confirmar que todos los ficheros necesarios (dataset, modelos) estan
# donde se espera antes de continuar
os.chdir('/content/drive/MyDrive/TFG/archive')
print(os.listdir('.'))

In [ ]:
# Comprobamos que los 12 ficheros de modelos exportados desde el notebook
# principal (pipeline_botiot_modelos.ipynb) estan todos presentes
print(os.listdir('modelos_exportados'))

In [ ]:
import joblib
from tensorflow import keras

# --- Carga de todos los modelos ya entrenados sobre BoT-IoT ---
# No se reentrena nada aqui: esta es la fase de validación externa, en la
# que aplicamos los modelos ya entrenados sobre tráfico real capturado
# por el honeypot, sin que hayan visto estos datos durante el entrenamiento

scaler = joblib.load('modelos_exportados/scaler.pkl')
le = joblib.load('modelos_exportados/label_encoder.pkl')

rf_bin = joblib.load('modelos_exportados/rf_bin.pkl')
rf_multi = joblib.load('modelos_exportados/rf_multi.pkl')
xgb_bin = joblib.load('modelos_exportados/xgb_bin.pkl')
xgb_multi = joblib.load('modelos_exportados/xgb_multi.pkl')
svm_bin = joblib.load('modelos_exportados/svm_bin.pkl')
svm_multi = joblib.load('modelos_exportados/svm_multi.pkl')

# Los autoencoders (Keras) se cargan con su propio metodo
encoder_b = keras.models.load_model('modelos_exportados/encoder_b.h5')
encoder_m = keras.models.load_model('modelos_exportados/encoder_m.h5')
xgb_ae_bin = joblib.load('modelos_exportados/xgb_ae_bin.pkl')
xgb_ae_multi = joblib.load('modelos_exportados/xgb_ae_multi.pkl')

# Mismas 9 caracteristicas usadas durante el entrenamiento; el orden
# importa porque el scaler y los modelos esperan las columnas en este
# orden exacto
features = ['pkts', 'bytes', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'srate', 'drate']

print("Todo cargado correctamente")

In [ ]:
import pandas as pd

# Cargamos el features.csv generado por el script zeek_to_botiot.py a
# partir del conn.log de Zeek: son los flujos de red reales capturados
# por el honeypot Cowrie, ya transformados al esquema de 9 caracteristicas
# de BoT-IoT (ver honeypot/zeek_to_botiot.py en el repositorio)
df_honeypot = pd.read_csv('/content/drive/MyDrive/TFG/features2.csv')
print(f"Total de flujos del honeypot: {len(df_honeypot)}")

In [ ]:
# --- Validacion externa: aplicamos los modelos sobre trafico real ---
# Escalamos las caracteristicas del honeypot con el mismo scaler que se
# ajusto sobre BoT-IoT (no se reajusta), para que ambas fuentes queden en
# la misma escala relativa que vieron los modelos durante el entrenamiento
X_honeypot = df_honeypot[features].values
X_honeypot_scaled = scaler.transform(X_honeypot)

# --- Modelos clasicos: clasificacion binaria (ataque vs normal) ---
pred_rf_bin  = rf_bin.predict(X_honeypot_scaled)
pred_xgb_bin = xgb_bin.predict(X_honeypot_scaled)
pred_svm_bin = svm_bin.predict(X_honeypot_scaled)

# --- Modelo hibrido: autoencoder + XGBoost ---
# Primero proyectamos al espacio latente con el encoder ya entrenado,
# despues clasificamos sobre esa representacion comprimida
X_honeypot_ae_b = encoder_b.predict(X_honeypot_scaled, verbose=0)
pred_ae_bin = xgb_ae_bin.predict(X_honeypot_ae_b)

# Como todo el trafico del honeypot es por definicion no autorizado
# (es un honeypot: ningun usuario legitimo se conecta a el), la metrica
# principal aqui es el RECALL: que proporcion de este trafico real se
# detecta correctamente como ataque
print("=== Recall sobre trafico real nunca visto (todo es 'ataque' por definicion) ===")
print(f"Random Forest        : {(pred_rf_bin==1).mean()*100:.2f}%  ({(pred_rf_bin==1).sum()} / {len(pred_rf_bin)})")
print(f"XGBoost              : {(pred_xgb_bin==1).mean()*100:.2f}%  ({(pred_xgb_bin==1).sum()} / {len(pred_xgb_bin)})")
print(f"SVM                  : {(pred_svm_bin==1).mean()*100:.2f}%  ({(pred_svm_bin==1).sum()} / {len(pred_svm_bin)})")
print(f"Autoencoder + XGBoost: {(pred_ae_bin==1).mean()*100:.2f}%  ({(pred_ae_bin==1).sum()} / {len(pred_ae_bin)})")

# --- Modelo multiclase: que categoria de ataque predice cada modelo ---
# No podemos calcular accuracy real aqui porque no tenemos la categoria
# verdadera de cada conexion (solo sabemos que TODO es ataque, no de que
# tipo exacto); esto es un analisis exploratorio de la distribucion de
# categorias predichas, util para contrastar con lo esperado
# teoricamente (mayoritariamente Reconnaissance, dado que el honeypot es pasivo)
pred_rf_multi = rf_multi.predict(X_honeypot_scaled)
pred_xgb_multi = xgb_multi.predict(X_honeypot_scaled)

print("\n=== Distribucion de categorias predichas (Random Forest multiclase) ===")
print(pd.Series(le.inverse_transform(pred_rf_multi)).value_counts())
print("\n=== Distribucion de categorias predichas (XGBoost multiclase) ===")
print(pd.Series(le.inverse_transform(pred_xgb_multi)).value_counts())